# Deterministic chaos in a four-component microbial chemostat

This notebook walks through the model of **Molz, Faybishenko & Agarwal (2019)**,
a chemostat holding a nutrient, two competing bacteria (rods and cocci), and a
predator that grazes on both. With the right dilution rate the three microbes
coexist forever but never repeat: the system is a low-dimensional example of
**deterministic chaos**, and it reproduces the chaos measured experimentally by
Becks *et al.* (2005, *Nature*).

The goal here is not just to draw pretty trajectories but to *establish* that the
dynamics are chaotic — with a positive Lyapunov exponent and the 0–1 test — and
to read off the **Lyapunov time**, which is the natural limit on how far ahead the
system can be predicted. That number is what a forecasting model ultimately has
to be scored against.

## The model

Four state variables in a well-mixed vessel fed at dilution rate $D$:

- $n$ — nutrient (mg/cc)
- $r$ — rod bacteria (cells/cc), strong competitor for nutrient
- $c$ — coccus bacteria (cells/cc), weaker competitor but less palatable
- $p$ — ciliate predator (cells/cc), eats both, prefers rods

Growth is Monod (Michaelis–Menten). Two features push the system past simple
oscillation into chaos: the predator's preference for rods **grows with rod
density**, and dead predator biomass is **recycled** into nutrient. The full
equations and all parameter values live in `src/chemostat.py`.

In [ ]:
import sys, os
sys.path.insert(0, "src")            # run this notebook from the repo root
import numpy as np
import matplotlib.pyplot as plt
import chemostat as cm

plt.rcParams.update({"figure.dpi": 110, "font.size": 11,
                     "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
COL = {"n": "#3b6ea5", "r": "#c1440e", "c": "#e0a800", "p": "#2a7f62"}

cm.Params()          # the parameter set, straight from Table 1.1 of the report

## 1. A chaotic trajectory

Integrate the equations at $D = 0.0207\,\text{hr}^{-1}$. All three microbes
persist, but every peak is a different height and the rhythm never locks in.
(A short run is used here for speed; `run_all.py` integrates much longer for the
publication figures.)

In [ ]:
t, Y = cm.simulate(cm.D_CHAOS, t_end=60_000, n_points=60_000)
n, r, c, p = Y
td = t / 24.0    # hours -> days

fig, ax = plt.subplots(4, 1, figsize=(9, 7), sharex=True)
for a, series, key, name in zip(ax, (n, r, c, p), "nrcp",
                                ("nutrient", "rods", "cocci", "predators")):
    a.plot(td, series, color=COL[key], lw=0.8)
    a.set_ylabel(name)
ax[-1].set_xlabel("time (days)")
ax[0].set_title(f"Chaotic coexistence at D = {cm.D_CHAOS}/hr")
plt.tight_layout()

## 2. The strange attractor

Plotting rods, cocci and predators against each other, the trajectory settles
onto a bounded, folded surface that it never exactly retraces — a strange
attractor. Order lives inside the apparent randomness.

In [ ]:
t, Y = cm.simulate(cm.D_CHAOS, t_end=150_000, n_points=150_000, discard_frac=0.15)
fig = plt.figure(figsize=(7.5, 6.5))
ax = fig.add_subplot(111, projection="3d")
ax.plot(Y[1], Y[2], Y[3], lw=0.3, color="#4b3b7a", alpha=0.8)
ax.set_xlabel("rods"); ax.set_ylabel("cocci"); ax.set_zlabel("predators")
ax.set_title("Strange attractor"); ax.view_init(elev=22, azim=-60)
plt.tight_layout()

## 3. The dilution rate selects the regime

Chaos is not automatic — it needs the flow rate to sit in a particular band.
Slower flow and the cocci wash out; faster flow and the rods wash out. Only in
between do all three survive, and there the motion is chaotic. This reproduces
the three cases in the paper.

In [ ]:
cases = [(cm.D_CHAOS, "chaos — all coexist"),
         (cm.D_COCCI_DIE, "steady — cocci wash out"),
         (cm.D_RODS_DIE, "steady — rods wash out")]

fig, ax = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
for a, (D, label) in zip(ax, cases):
    t, Y = cm.simulate(D, t_end=60_000, n_points=60_000)
    td = t / 24.0
    a.plot(td, Y[1], color=COL["r"], lw=0.7, label="rods")
    a.plot(td, Y[2], color=COL["c"], lw=0.7, label="cocci")
    a.plot(td, Y[3] * 100, color=COL["p"], lw=0.7, label="predators ×100")
    a.set_yscale("symlog", linthresh=1e3); a.set_ylabel("cells/cc")
    a.set_title(f"D = {D}/hr  —  {label}", fontsize=10)
    print(f"D={D}: {cm.survivors(Y)}")
ax[0].legend(fontsize=8, loc="upper right")
ax[-1].set_xlabel("time (days)")
plt.tight_layout()

## 4. Sensitivity to initial conditions

The fingerprint of chaos: nudge the starting nutrient by one part in $10^8$ and
the two runs stay together for a while, then peel apart and end up completely out
of phase. The separation grows exponentially until it saturates at the size of
the attractor — and that saturation time is the hard ceiling on prediction.

In [ ]:
D = cm.D_CHAOS
_, Y0 = cm.simulate(D, t_end=8_000, n_points=4_000)
ya = Y0[:, -1].copy()
yb = ya.copy(); yb[0] += 1e-8                     # perturb nutrient

ta, Ya = cm.simulate(D, t_end=25_000, n_points=25_000, y0=ya)
tb, Yb = cm.simulate(D, t_end=25_000, n_points=25_000, y0=yb)
scale = np.array([0.05, 1e6, 1e6, 3e3])
sep = np.linalg.norm(((Ya - Yb).T / scale), axis=1)
td = ta / 24.0

fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax[0].plot(td, Ya[1], color=COL["r"], lw=0.7, label="A")
ax[0].plot(td, Yb[1], color="#6a2a0a", lw=0.7, ls="--", label="B")
ax[0].set_ylabel("rods"); ax[0].legend(fontsize=8, loc="upper right")
ax[0].set_title("Two runs, initial nutrient differing by 1e-8 mg/cc")
ax[1].semilogy(td, sep, color="#333", lw=0.8)
ax[1].set_ylabel("separation"); ax[1].set_xlabel("time (days)")
plt.tight_layout()

## 5. Quantifying the chaos

Two independent tests. The **largest Lyapunov exponent** is found by propagating
the tangent-linear equation alongside the trajectory and measuring the average
stretching rate; a positive value means nearby states diverge, i.e. chaos. Its
reciprocal, the **Lyapunov time**, is the timescale over which prediction stays
meaningful. The **0–1 test** (Gottwald–Melbourne) returns $K \approx 1$ for chaos
and $K \approx 0$ for regular motion, using no derivatives at all.

(The Lyapunov calculation below is shortened for the notebook; `run_all.py` runs
it longer for a tighter estimate.)

In [ ]:
lam = cm.largest_lyapunov(cm.D_CHAOS, t_run=80_000)
t, Y = cm.simulate(cm.D_CHAOS, t_end=150_000, n_points=150_000, discard_frac=0.4)
K = cm.zero_one_test(Y[1])

print(f"largest Lyapunov exponent  lambda = {lam:.5f} / hr")
print(f"Lyapunov time  1/lambda           = {1/lam:.0f} hr  (~{1/lam/24:.0f} days)")
print(f"0-1 test statistic  K             = {K:.3f}")

## 6. Chaos and order interleaved

Sweeping the dilution rate and recording the rod density on a Poincaré section
gives a bifurcation diagram. Where the points spread into a smear, the motion is
chaotic; where they collapse onto a few values, a periodic window has opened.
The full, dense version is `figures/04_bifurcation.png`; a coarse scan here shows
the idea without the wait.

In [ ]:
Ds = np.linspace(0.0204, 0.0214, 22)     # coarse; the saved figure uses ~78 points
xs, ys = [], []
for D in Ds:
    rc = cm.poincare_r(D, t_end=80_000, n_points=120_000, discard_frac=0.4)
    if rc.size:
        xs.extend([D] * min(rc.size, 60)); ys.extend(rc[-60:])

plt.figure(figsize=(8, 5))
plt.plot(xs, np.array(ys) / 1e6, ".", ms=2, color="#222", alpha=0.5)
plt.axvline(cm.D_CHAOS, color="#c1440e", ls="--", lw=1)
plt.xlabel("dilution rate D (1/hr)")
plt.ylabel("rod density at section (1e6 cells/cc)")
plt.title("Bifurcation diagram (coarse)")
plt.tight_layout()

## Where this goes next

A clean chaos generator with a known Lyapunov time turns "can we forecast this?"
into a measurable question. The planned next stages:

1. Use `simulate()` to build a library of trajectories across the chaotic band,
   with realistic sampling and added observational noise.
2. Train and compare sequence models — an LSTM baseline against reservoir
   computing — to predict the multivariate state forward.
3. Report the forecast horizon in **Lyapunov times**, the only fair clock for a
   chaotic system.